# 14 - Quantum Phase Estimation

Estimate the eigenvalue of a unitary operator.

**Concepts:** Phase estimation, inverse QFT, eigenvalues

In [ ]:
import quantsdk as qs
import math

## Phase Estimation for T Gate

The T gate has eigenvalue $e^{i\pi/4}$, so the phase is $\phi = 1/8$.

With 3 counting qubits we can represent $\phi = 0.001_2 = 1/8$ exactly.

In [ ]:
n_count = 3  # counting qubits
circuit = qs.Circuit(n_count + 1, name="QPE-T")

# Prepare eigenstate |1> on target qubit
circuit.x(n_count)

# Hadamard on counting qubits
for i in range(n_count):
    circuit.h(i)

# Controlled-U^(2^k) operations (textbook convention: qubit 0 = MSB)
# T gate: phase = pi/4, so T^(2^k) = Phase(pi/4 * 2^k)
for k in range(n_count):
    power = n_count - 1 - k  # qubit 0 gets highest power
    angle = math.pi / 4 * (2 ** power)
    circuit.cp(k, n_count, angle)

# Inverse QFT on counting qubits
circuit.swap(0, 2)
circuit.h(0)
circuit.cp(0, 1, -math.pi / 2)
circuit.h(1)
circuit.cp(0, 2, -math.pi / 4)
circuit.cp(1, 2, -math.pi / 2)
circuit.h(2)

# Measure counting qubits
for i in range(n_count):
    circuit.measure(i)

result = qs.run(circuit, shots=1000, seed=42)
print(f"QPE for T gate (phase = 1/8):")
print(f"Result: {result.counts}")
print(f"Most likely: {result.most_likely}")

# Convert binary to phase
measured = result.most_likely
phase = int(measured, 2) / (2 ** n_count)
print(f"Estimated phase: {phase} (expected: 0.125 = 1/8)")

## Phase Estimation for S Gate

S gate: eigenvalue $e^{i\pi/2}$, phase $\phi = 1/4$.

In [ ]:
n_count = 3
circuit = qs.Circuit(n_count + 1, name="QPE-S")

circuit.x(n_count)
for i in range(n_count):
    circuit.h(i)

# S gate: phase = pi/2
for k in range(n_count):
    power = n_count - 1 - k
    angle = math.pi / 2 * (2 ** power)
    circuit.cp(k, n_count, angle)

# Inverse QFT
circuit.swap(0, 2)
circuit.h(0)
circuit.cp(0, 1, -math.pi / 2)
circuit.h(1)
circuit.cp(0, 2, -math.pi / 4)
circuit.cp(1, 2, -math.pi / 2)
circuit.h(2)

for i in range(n_count):
    circuit.measure(i)

result = qs.run(circuit, shots=1000, seed=42)
measured = result.most_likely
phase = int(measured, 2) / (2 ** n_count)
print(f"QPE for S gate: measured={measured}, phase={phase} (expected: 0.25)")